# 02 — Stage 1 Planner fine-tune

QLoRA fine-tune of `Qwen2.5-7B-Instruct` on the `(english -> pseudocode)`
pairs in `data/stage1_planner/englishtopseudo.jsonl`, after registering the
DSL special tokens (`<PLAN>`, `</PLAN>`, `<STEP>`) from `src/dsl/schema.py`
into the tokenizer — so the model emits them as single tokens instead of
spelling them out character by character.

**This notebook is self-contained: `Runtime -> Run all` on a GPU runtime is
the whole procedure.** Section 2 clones the repo, mounts Drive, and checks
the GPU for you; there are no cells to add by hand. Each section says what
you should see in its output, so you can tell a good run from a bad one as
you go.

Two things worth doing *before* you connect to a runtime, because compute
units are billed on GPU-connected wall time:

1. Edit the section 1 config cell. Colab lets you edit cell source with no
   runtime attached — only *running* needs one.
2. `File -> Save a copy in Drive`, so your edits survive. Opened from the
   GitHub link, this notebook is a read-only render.

`docs/colab_setup.md` in the repo has the rest: choosing a GPU, what
persists between sessions, troubleshooting, and how to reuse the adapter
afterwards.

## 1. Config

Everything you might want to change lives here. Paths are absolute and
derived from `REPO_DIR`, so nothing depends on the runtime's working
directory.

| Setting | Default | Notes |
| --- | --- | --- |
| `REPO_BRANCH` | `main` | Change if the data you want isn't merged yet. |
| `OUTPUT_DIR` | `stage1_planner_qlora` | Runtime-local, so epoch checkpoints die with the session. Point it inside `DRIVE_CHECKPOINT_DIR` if you expect interruptions and want to resume with `trainer.train(resume_from_checkpoint=True)`. |
| `CACHE_MODEL_ON_DRIVE` | `False` | `True` keeps the ~15GB base model on Drive so later sessions skip the download. Only worth it if your Drive has the space to spare. |
| `MAX_SEQ_LEN` | 512 | Prompt + plan, truncated. Raise if you add long plans. |
| `NUM_EPOCHS` | 15 | High on purpose — 50 examples is a proof of concept. |
| `LEARNING_RATE` | 2e-4 | Standard LoRA range (1e-4 to 3e-4). |
| `PER_DEVICE_BATCH_SIZE` × `GRAD_ACCUM_STEPS` | 4 × 4 | Effective batch 16, ~3 steps per epoch. |

If you hit CUDA out of memory later, drop `PER_DEVICE_BATCH_SIZE` to 2 or 1
and raise `GRAD_ACCUM_STEPS` to keep the effective batch at 16.

In [ ]:
# Where the code and data come from
REPO_URL = "https://github.com/ashfordreyes/pythonllm.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/pythonllm"

DATA_PATH = f"{REPO_DIR}/data/stage1_planner/englishtopseudo.jsonl"
SCHEMA_PATH = f"{REPO_DIR}/src/dsl/schema.py"

# Where results go. Only Drive survives the runtime shutting down.
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/pythonllm_checkpoints"
OUTPUT_DIR = "stage1_planner_qlora"
CACHE_MODEL_ON_DRIVE = False

# Training
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LEN = 512
NUM_EPOCHS = 15
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4

## 2. Setup

Four cells that turn a bare runtime into one this notebook can train on:
check the GPU, install the libraries, mount Drive, clone the repo. Run them
in order — the Drive mount has to happen before anything imports
`transformers` for `CACHE_MODEL_ON_DRIVE` to take effect.

Once these have run clean you don't touch them again for the session.

### 2a. Check the GPU

Colab silently falls back to whatever hardware is free, so confirm you got
what you asked for. Expect ~40GB total memory on an A100 or ~23GB on an L4,
and `True` for both torch checks.

`is_bf16_supported()` must be `True`: section 7 trains with `bf16=True`. A
free T4 reports `False` here — see `docs/colab_setup.md` for why that GPU
isn't practical for this run.

In [ ]:
!nvidia-smi

import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())  # must be True

### 2b. Install dependencies

`bitsandbytes` provides the 4-bit (NF4) quantization and `peft` provides
LoRA. Takes 1-2 minutes.

If pip reports that it upgraded an already-imported package (usually `torch`
or `transformers`), restart the runtime and re-run from section 1 —
otherwise you get version-mismatch errors several cells later.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets trl

### 2c. Mount Google Drive

**Nothing under `/content` survives the session** — not the clone, not the
model download, not `OUTPUT_DIR`. Drive is the only durable storage Colab
gives you, so this cell mounts it and creates the checkpoint folder that
section 9 copies the trained adapter into.

The first run opens a Google auth popup; approve it. Re-running when already
mounted is harmless.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

if CACHE_MODEL_ON_DRIVE:
    # Must be set before transformers is imported, or it is ignored.
    os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
    print("HF cache on Drive:", os.environ["HF_HOME"])

print("Checkpoints will go to:", DRIVE_CHECKPOINT_DIR)

### 2d. Clone the repo

Opening this notebook does *not* bring the repo with it. The cells below
read the dataset off the runtime's disk and import `SPECIAL_TOKENS` from
`src/dsl/schema.py`, so the files have to be there.

`ashfordreyes/pythonllm` is public, so no token is needed. The cell is safe
to re-run: it pulls instead of cloning if the directory already exists.

Expect `OK: 50 examples`. If the assert fires instead, the clone failed or
`REPO_BRANCH` names a branch that doesn't have the data — fix that before
going on, because the next section is where a missing clone would otherwise
surface as `ModuleNotFoundError: No module named 'dsl'`.

In [ ]:
import pathlib
import sys

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already present; pulling")
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}

for path in (DATA_PATH, SCHEMA_PATH):
    assert pathlib.Path(path).exists(), (
        f"missing {path} — the clone failed, or branch '{REPO_BRANCH}' "
        "doesn't have this file"
    )

sys.path.insert(0, f"{REPO_DIR}/src")

n_examples = sum(1 for _ in open(DATA_PATH))
print(f"OK: {n_examples} examples in {DATA_PATH}")

## 3. Tokenizer and DSL special tokens

`SPECIAL_TOKENS` is imported straight from `src/dsl/schema.py` so the
tokenizer always matches the DSL definition in the repo, rather than
hardcoding the token strings here.

Expect `Added 3 new special tokens`. A `0` means either they were already in
the vocab or `SPECIAL_TOKENS` came back empty — worth checking before you
spend a 15GB model download on it, which is why the import and the tokenizer
share a cell.

In [ ]:
from dsl.schema import SPECIAL_TOKENS
from transformers import AutoTokenizer

print("DSL special tokens:", SPECIAL_TOKENS)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
num_added = tokenizer.add_special_tokens(
    {"additional_special_tokens": SPECIAL_TOKENS}
)
print(f"Added {num_added} new special tokens; vocab size now {len(tokenizer)}")

## 4. Load the base model in 4-bit and resize embeddings

Downloads ~15GB (3-10 minutes; the download is bf16, only the in-memory
weights are 4-bit) and loads it quantized.

The 3 new tokens grew the tokenizer, so the embedding matrix is resized to
match *before* LoRA wraps the model. Skip that and the first `<PLAN>` token
id lookup errors out.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
)
model.resize_token_embeddings(len(tokenizer))
model.config.use_cache = False

## 5. LoRA setup

Rank 16, alpha 32, dropout 0.05, applied to all attention and MLP
projections. Expect roughly 40M trainable parameters out of ~7.6B — well
under 1%.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Load and format the dataset

Each example is formatted with the base model's own chat template so the
planner learns to respond to an instruction-style prompt with a pseudocode
plan, then loss-masked so only the pseudocode completion — not the prompt —
contributes to the loss.

Expect the dataset summary to show 50 rows, and the first row to have
`english` and `pseudocode` keys.

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("json", data_files=DATA_PATH, split="train")
print(raw_dataset)
print(raw_dataset[0])

In [ ]:
SYSTEM_PROMPT = (
    "You are a planner that turns a task description into a pseudocode plan "
    "using the pythonllm DSL. Respond with only the plan, wrapped in "
    "<PLAN>...</PLAN> and made of <STEP> lines."
)


def format_example(example):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["english"]},
    ]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    full_text = prompt_text + example["pseudocode"] + tokenizer.eos_token

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )

    labels = list(full["input_ids"])
    prompt_len = min(len(prompt_ids), len(labels))
    for i in range(prompt_len):
        labels[i] = -100

    full["labels"] = labels
    return full


tokenized_dataset = raw_dataset.map(
    format_example, remove_columns=raw_dataset.column_names
)

# Sanity check the masking: this should print the plan and nothing else.
row = tokenized_dataset[0]
supervised = [t for t, l in zip(row["input_ids"], row["labels"]) if l != -100]
print(f"{len(supervised)} of {len(row['labels'])} tokens supervised")
print(tokenizer.decode(supervised))

## 7. Train

15 epochs over 50 examples is ~45 steps — a few minutes on an A100 or L4.
Loss is logged every step and should fall.

If it stays flat or goes `nan`: flat usually means an example got fully
masked (check the supervised-token count printed above is not 0), and `nan`
usually means bf16 isn't really supported — see the section 2a output.

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

data_collator = DataCollatorForSeq2Seq(
    tokenizer, padding=True, label_pad_token_id=-100
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    bf16=True,
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

## 8. Save the adapter and tokenizer

Both, together: the tokenizer carries the DSL special tokens, and an adapter
loaded against a stock-vocab tokenizer won't work.

In [ ]:
FINAL_DIR = f"{OUTPUT_DIR}/final"
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"Saved LoRA adapter + tokenizer to {FINAL_DIR}")

## 9. Copy the adapter to Google Drive

**Do not skip this.** `FINAL_DIR` is on the runtime's disk, which is
destroyed when the session ends, disconnects, or is reclaimed for idling.
This is the one artifact of the whole run you can't cheaply reproduce.

Drive is already mounted from section 2c.

In [ ]:
DRIVE_ADAPTER_DIR = f"{DRIVE_CHECKPOINT_DIR}/stage1_planner"

!rm -rf {DRIVE_ADAPTER_DIR}
!cp -r {FINAL_DIR} {DRIVE_ADAPTER_DIR}
!ls {DRIVE_ADAPTER_DIR}
print(f"Adapter saved to {DRIVE_ADAPTER_DIR}")

## 10. Quick sanity check

Runs one held-in example through the fine-tuned model. It decodes with
`skip_special_tokens=False`, so you should literally see `<PLAN>`, `<STEP>`
and `</PLAN>` in the output.

If the model spells out `<`, `PLAN`, `>` as separate pieces, the tokenizer
in use is missing the special tokens — load the one saved next to the
adapter, not a fresh one from the hub.

In [ ]:
model.eval()
test_english = raw_dataset[0]["english"]
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_english},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        inputs, max_new_tokens=MAX_SEQ_LEN, do_sample=False
    )

print("INPUT:", test_english)
print("OUTPUT:", tokenizer.decode(output_ids[0][inputs.shape[1]:], skip_special_tokens=False))

## 11. Release the runtime

Compute units are billed on GPU-connected wall time, and closing the browser
tab does **not** disconnect — Colab keeps the runtime alive in the
background, still billing. This cell ends the session and frees the GPU.

It kills the kernel, so only run it once section 9 has confirmed the adapter
is on Drive. Comment it out if you still want to poke at the in-memory
model.

In [ ]:
from google.colab import runtime

runtime.unassign()